In [2]:
import numpy as np

In [ ]:
def householder_qr_simple(A_init, b_init, eps=1e-12):
    A = A_init.copy().astype(float)
    b = b_init.copy().astype(float)
    n = len(A)
    Q = np.eye(n)

    for r in range(n - 1):
        sigma = 0.0
        for i in range(r, n):
            sigma += A[i, r] ** 2

        if np.sqrt(sigma) <= eps:
            print(f"Matrix nearly singular at column {r}")
            break
      
        if A[r, r] >= 0:
            sign = 1.0 
        else: 
            sign =-1.0

        k = -sign * np.sqrt(sigma)
        if abs(k) < eps:
            k = -np.sqrt(sigma)

        u = np.zeros(n)
        u[r] = A[r, r] - k
        for i in range(r + 1, n):
            u[i] = A[i, r]

        norm_u_sq = 0.0
        for i in range(r, n):
            norm_u_sq += u[i] ** 2
            
        if norm_u_sq < eps:
            beta = 0.0
        else:
            beta = 2.0 / norm_u_sq

       
        for j in range(r, n):
            gamma = 0.0
            for i in range(r, n):
                gamma += u[i] * A[i, j]
            gamma *= beta
            for i in range(r, n):
                A[i, j] -= gamma * u[i]

       
        gamma_b = 0.0
        for i in range(r, n):
            gamma_b += u[i] * b[i]
        gamma_b *= beta
        for i in range(r, n):
            b[i] -= gamma_b * u[i]

       
        for j in range(n):
            gamma_Q = 0.0
            for i in range(r, n):
                gamma_Q += u[i] * Q[i, j]
            gamma_Q *= beta
            for i in range(r, n):
                Q[i, j] -= gamma_Q * u[i]

        
        A[r, r] = k

        for i in range(r + 1, n):
            A[i, r] = 0.0

    
    return Q.T, A, b


In [ ]:
def back_substitution(R, y, eps=1e-12):
    n = len(y)
    x = np.zeros(n)
    for i in reversed(range(n)):
        s = 0.0
        for j in range(i + 1, n):
            s += R[i, j] * x[j]

        if abs(R[i, i]) < eps:
            raise ValueError("Singular or ill-conditioned matrix during back substitution.")
        
        x[i] = (y[i] - s) / R[i, i]
    return x

In [ ]:
def inverse_householder(A, Q, R, eps=1e-12):
    n = len(A)
    A_inv = np.zeros((n, n))
    
    for j in range(n):
        e = np.zeros(n)
        e[j] = 1.0
        b = Q.T @ e
        x = back_substitution(R, b)
        for i in range(n):
            A_inv[i, j] = x[i]
            
    return A_inv

In [11]:

np.set_printoptions(precision=6, suppress=True)
  
n = int(input("Enter matrix dimension n: ") or 4)
eps = 1e-8
np.random.seed(42)  

A = np.random.rand(n, n)
s = np.random.rand(n)
b = np.zeros(n)
for i in range(n):  
    for j in range(n):
        b[i] += A[i, j] * s[j]

print("\nMatrix A:\n", A)
print("Vector s:", s)
print("Vector b = A*s:", b)

   
Q, R, b_trans = householder_qr_simple(A, b, eps)
x_H = back_substitution(R, Q.T @ b)
Q_lib, R_lib = np.linalg.qr(A)
x_QR = np.linalg.solve(R_lib, Q_lib.T @ b)

err_x_diff = np.linalg.norm(x_QR - x_H, 2)
err_Axb_H = np.linalg.norm(A @ x_H - b, 2)
err_Axb_QR = np.linalg.norm(A @ x_QR - b, 2)
err_xs_H = np.linalg.norm(x_H - s, 2) / np.linalg.norm(s, 2)
err_xs_QR = np.linalg.norm(x_QR - s, 2) / np.linalg.norm(s, 2)


A_inv_H = inverse_householder(A, Q, R)
A_inv_lib = np.linalg.inv(A)
err_inv = np.linalg.norm(A_inv_H - A_inv_lib, 2)


print("\n--- Results ---")
print("x_Householder =", x_H)
print("x_QR (library) =", x_QR)
print("||x_QR - x_H||2 =", err_x_diff)
print("\nErrors:")
print("||A x_H - b||2 =", err_Axb_H)
print("||A x_QR - b||2 =", err_Axb_QR)
print("||x_H - s|| / ||s|| =", err_xs_H)
print("||x_QR - s|| / ||s|| =", err_xs_QR)
print("\nInverse comparison:")
print("||A^-1_H - A^-1_lib||2 =", err_inv)





Matrix A:
 [[0.37454  0.950714 0.731994 0.598658 0.156019 0.155995]
 [0.058084 0.866176 0.601115 0.708073 0.020584 0.96991 ]
 [0.832443 0.212339 0.181825 0.183405 0.304242 0.524756]
 [0.431945 0.291229 0.611853 0.139494 0.292145 0.366362]
 [0.45607  0.785176 0.199674 0.514234 0.592415 0.04645 ]
 [0.607545 0.170524 0.065052 0.948886 0.965632 0.808397]]
Vector s: [0.304614 0.097672 0.684233 0.440152 0.122038 0.495177]
Vector b = A*s: [1.067589 1.308046 0.776426 0.857137 0.673878 1.18203 ]

--- Results ---
x_Householder = [0.304614 0.097672 0.684233 0.440152 0.122038 0.495177]
x_QR (library) = [0.304614 0.097672 0.684233 0.440152 0.122038 0.495177]
||x_QR - x_H||2 = 1.1857187100668868e-15

Errors:
||A x_H - b||2 = 5.551115123125783e-16
||A x_QR - b||2 = 2.482534153247273e-16
||x_H - s|| / ||s|| = 3.0172879712057405e-15
||x_QR - s|| / ||s|| = 2.2439505337097036e-15

Inverse comparison:
||A^-1_H - A^-1_lib||2 = 4.33390353321907e-15
